In [1]:
import os
import sys

os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jdk-25.0.4.1"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["PATH"] = os.environ["JAVA_HOME"] + r"\bin;" + os.environ["PATH"]

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[2]") \
    .appName("Experiment9") \
    .config("spark.pyspark.python", sys.executable) \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("Spark version:", spark.version)

Spark version: 4.2.0


In [5]:
from pyspark.sql import SparkSession
from pyspark.ml.clustering import KMeans
from pyspark.ml.feature import VectorAssembler
import random

data_points = []

for _ in range(100):
    data_points.append((
        10 + random.uniform(-2, 2),
        10 + random.uniform(-2, 2)
    ))

for _ in range(100):
    data_points.append((
        50 + random.uniform(-2, 2),
        50 + random.uniform(-2, 2)
    ))

df = spark.createDataFrame(data_points, ["feature1", "feature2"])

print("Generated Data:")
df.show(5)

Generated Data:
+------------------+-----------------+
|          feature1|         feature2|
+------------------+-----------------+
|10.998398215521188|9.706914831318832|
| 9.279216014067652|9.153145801912803|
|10.650301698919984|8.387050024516881|
| 9.834465872711846|8.029524233916172|
|11.139216507013138|9.937050828215249|
+------------------+-----------------+
only showing top 5 rows


In [4]:
assembler = VectorAssembler(
    inputCols=["feature1", "feature2"],
    outputCol="features"
)

transformed_df = assembler.transform(df)

print("Transformed Data with 'features' column:")
transformed_df.show(5)

Transformed Data with 'features' column:
+------------------+------------------+--------------------+
|          feature1|          feature2|            features|
+------------------+------------------+--------------------+
|11.822261989604655|11.129982073321518|[11.8222619896046...|
|  9.62680043020975|10.324000270294775|[9.62680043020975...|
| 8.525678671553877|     8.13954975667|[8.52567867155387...|
| 8.234891364217727|11.299067526835938|[8.23489136421772...|
| 8.741644108725131|  11.8624940184442|[8.74164410872513...|
+------------------+------------------+--------------------+
only showing top 5 rows


In [6]:
kmeans = KMeans(k=2, seed=1)

model = kmeans.fit(transformed_df)

predictions = model.transform(transformed_df)

print("Predictions with 'prediction' (cluster) column:")
predictions.show(5)

Predictions with 'prediction' (cluster) column:
+------------------+------------------+--------------------+----------+
|          feature1|          feature2|            features|prediction|
+------------------+------------------+--------------------+----------+
|11.822261989604655|11.129982073321518|[11.8222619896046...|         1|
|  9.62680043020975|10.324000270294775|[9.62680043020975...|         1|
| 8.525678671553877|     8.13954975667|[8.52567867155387...|         1|
| 8.234891364217727|11.299067526835938|[8.23489136421772...|         1|
| 8.741644108725131|  11.8624940184442|[8.74164410872513...|         1|
+------------------+------------------+--------------------+----------+
only showing top 5 rows


In [7]:
cost = model.summary.trainingCost

print("Within Set Sum of Squared Errors (WSSSE):", cost)

centers = model.clusterCenters()

print("Cluster Centers:")
for center in centers:
    print(center)

Within Set Sum of Squared Errors (WSSSE): 490.54023376785324
Cluster Centers:
[49.84736317 50.08805307]
[9.93870832 9.71786326]
